# 01 — Dataset audit: MODE A vs MODE B per folder

**What this notebook does.** Opens every dataset folder under `DATA_ROOT`,
infers each folder's own image/mask naming convention, and works out from
the pixels whether its masks paint boundaries as a colour (MODE A) or are
phase-label maps (MODE B). It records the evidence for each verdict, not
just the verdict, and writes `reports/audit.json` and `reports/audit.md`.

**What must already exist.**

- `GH_TOKEN` as a host secret, and the datasets at the location named in
  `configs/default.yaml` — i.e. `notebooks/00_bootstrap.ipynb` passes

**What it produces.** `reports/audit.json` (machine-readable; every later
step reads a folder's mode from here and must never assume it) and
`reports/audit.md` (the same content for a human to disagree with). Both are
committed back to the repo by the last cell.

**Expected runtime on a free T4.** 3–6 minutes. Pairing is counted over
every file; pixel statistics come from a fixed random sample of pairs per
folder (`src.audit.SAMPLE_SIZE`, seed 0), because skeletonising ~1900
full-size masks would take far longer and change no verdict. No GPU is used —
the audit is pure NumPy and scikit-image.

## Cell 1 — the standard bootstrap block

Identical in every notebook. It reads `GH_TOKEN` from the host secret store,
fetches `scripts/bootstrap_session.py` through the GitHub API, and hands over
to `bootstrap()`, which clones the repo, installs what is missing, mounts
Drive on Colab, verifies `DATA_ROOT`, and returns `PATHS`. Re-running it
after a disconnect is the correct way to recover.

In [ ]:
# --- standard bootstrap block: identical in every notebook ---------------
OWNER, REPO, BRANCH = "arhorri", "boundary", "main"

import importlib, os, pathlib, sys, urllib.request


def _gh_token():
    """Read GH_TOKEN from whichever secret store this host provides."""
    try:
        from google.colab import userdata

        return userdata.get("GH_TOKEN")
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("GH_TOKEN")
    except Exception:
        pass
    return os.environ.get("GH_TOKEN")


_token = _gh_token()
if not _token:
    raise SystemExit(
        "GH_TOKEN secret is missing.\n"
        "  Colab : key icon in the left sidebar -> add GH_TOKEN -> notebook access ON\n"
        "  Kaggle: Add-ons -> Secrets -> add GH_TOKEN -> attach to this notebook"
    )

_req = urllib.request.Request(
    f"https://api.github.com/repos/{OWNER}/{REPO}/contents/scripts/bootstrap_session.py?ref={BRANCH}",
    headers={
        "Authorization": f"Bearer {_token}",
        "Accept": "application/vnd.github.raw",
    },
)
pathlib.Path("bootstrap_session.py").write_bytes(urllib.request.urlopen(_req).read())
del _token

if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))
import bootstrap_session

bootstrap_session = importlib.reload(bootstrap_session)

PATHS = bootstrap_session.bootstrap(
    repo_url=f"https://github.com/{OWNER}/{REPO}.git", branch=BRANCH
)

## Why MODE A vs MODE B is the most important fact about a folder

The goal of Phase 0 is a network that predicts **phase boundaries and grain
boundaries** from a raw micrograph. The labels for that network are derived
from each dataset's masks — and the masks come in two incompatible kinds.

**MODE A — boundaries are painted.** Someone drew the boundary network into
the mask as its own distinct colour, usually a saturated one, one to three
pixels wide. Extraction is HSV colour thresholding: select that colour and
you have the label. What you get includes *intra-phase grain boundaries* —
the lines between two grains of the *same* phase — because the annotator drew
them. That is the signal this project actually wants.

**MODE B — the mask is a phase-label map.** Each pixel carries the identity
of the phase it belongs to, and nothing else. There is no painted line.
Extraction is `find_boundaries` on the quantized label map, which returns
exactly the pixels where one phase meets another: *phase interfaces only*.
Two adjacent grains of the same phase carry the same label, so the boundary
between them does not exist in the file. **It cannot be recovered by any
amount of processing** — the information was never stored. Inventing it would
mean putting a learned model in the label-generation path, which this project
forbids.

So the mode decides what a folder's ground truth *can* teach:

| | MODE A | MODE B |
| --- | --- | --- |
| extraction | HSV threshold on the painted colour | `find_boundaries` on labels |
| phase interfaces | yes | yes |
| grain boundaries | yes | **no — absent from the data** |
| typical boundary pixel fraction | the painted line's own fraction | thin interface skeleton |

A MODE B folder trained as if it were MODE A teaches the network that grain
boundaries are *not* boundaries — actively wrong supervision, worse than
leaving the folder out. That is why the mode is measured here, written to
`reports/audit.json`, and read from there by every later step.

**How the verdict is reached.** For every colour in a mask that covers
between 0.05% and 30% of the pixels, the audit measures three things: mean
thickness (area ÷ skeleton length — a drawn line is 1–3 px thick, a phase
region is tens), how much of the frame the colour's bounding box spans (a
boundary network reaches everywhere; an artefact does not), and how many
sampled masks the same colour qualifies in. MODE A needs a colour that is
sparse, thin *and* frame-spanning in at least 60% of the sample. Everything
else is MODE B. All thresholds are constants at the top of `src/audit.py`
and are copied into the report so a verdict can be re-judged without re-running.

## Run the audit

All of the work lives in `src/audit.py`; this cell only supplies the paths
resolved by the bootstrap and a `tqdm` progress bar. Pairing (how many
images, masks, matched pairs, and which filenames matched nothing) is counted
over *every* file in the folder. The pixel statistics — palette, thickness,
connected components, boundary fractions — come from a fixed random sample of
pairs per folder so that the audit stays inside a session's lifetime.

A folder that cannot be audited at all does not abort the run: it is recorded
under `failures` in the report and the checks cell below turns it into a FAIL.

In [ ]:
from pathlib import Path

from tqdm.auto import tqdm

from src import audit

data_root = Path(PATHS["data_root"])
reports_dir = Path(PATHS["reports_dir"])


def progress(seq, desc=""):
    return tqdm(seq, desc=desc, leave=False, unit="pair")


report = audit.audit_datasets(
    data_root,
    sample_size=audit.SAMPLE_SIZE,
    seed=audit.SAMPLE_SEED,
    progress=progress,
)
json_path, md_path = audit.write_reports(report, reports_dir)
print(f"wrote {json_path}")
print(f"wrote {md_path}")
if report["failures"]:
    print("\nfolders that could not be audited:")
    for name, err in report["failures"].items():
        print(f"  {name}: {err}")

## The verdict table

One row per folder: the mode, the pairing counts, how thin and how sparse the
MODE A candidate colour was, and the boundary pixel fraction *each* mode would
produce. Both fractions are shown for every folder, including MODE B ones,
because that comparison is what makes a verdict checkable: in a MODE A folder
the painted-colour fraction is a real, thin network; in a MODE B folder there
is no such colour and only the `find_boundaries` column is meaningful.

Read the `reason` column as the one-line justification, and the sample images
below as the visual cross-check.

In [ ]:
import pandas as pd

rows = []
for name, d in report["datasets"].items():
    ev = d["mode"]["evidence"]
    rows.append({
        "folder": name,
        "MODE": d["mode"]["inferred"],
        "pairs": d["pairing"]["n_pairs"],
        "unmatched img": len(d["pairing"]["unmatched_images"]),
        "unmatched mask": len(d["pairing"]["unmatched_masks"]),
        "mask colours": d["palette"]["n_unique_colours"]["median"],
        "quantized": d["palette"]["quantized"],
        "cand colour": ev["candidate_colour"],
        "cand frac": ev["pixel_fraction"]["median"],
        "thickness px": ev["mean_thickness_px"]["median"],
        "votes": f"{ev['files_qualifying']}/{ev['files_sampled']}",
        "bnd frac A": d["boundary_fraction"]["mode_a"]["median"],
        "bnd frac B": d["boundary_fraction"]["mode_b"]["median"],
        "majority": d["mode_b"]["majority_structure"],
        "reason": d["mode"]["reason"],
    })

verdicts = pd.DataFrame(rows).set_index("folder")
pd.set_option("display.max_colwidth", 90)
display(verdicts.drop(columns=["reason"]))
for name, reason in verdicts["reason"].items():
    print(f"{name}: {reason}")

## Check the verdict by eye

The numbers can be right and still describe the wrong thing, so every folder
gets two sampled pairs drawn here: the raw micrograph, the mask as stored,
and the mask's colour histogram (each bar drawn in its own colour, log scale,
since a painted boundary is by definition a rare colour).

What to look for. **MODE A**: the mask shows a visible line network in one
distinct colour, and the histogram has a small bar in that colour far below
the phase colours. **MODE B**: the mask is flat regions of colour meeting
each other with no drawn line, and every histogram bar is a phase. If a
folder's picture disagrees with its verdict, the thresholds at the top of
`src/audit.py` are what to argue with.

Images are drawn at their stored resolution with nearest-neighbour
interpolation — masks must never be smoothed, and nothing here is resized.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

N_SAMPLES = 2

for name, d in report["datasets"].items():
    pairing = audit.discover_pairs(Path(d["path"]))
    pairs = pairing["pairs"][:N_SAMPLES]
    fig, axes = plt.subplots(len(pairs), 3, figsize=(15, 4.6 * len(pairs)),
                             squeeze=False)
    fig.suptitle(f"{name} — inferred MODE {d['mode']['inferred']} "
                 f"({pairing['convention']})", fontsize=13)

    for row, (img_path, mask_path) in enumerate(pairs):
        image = audit.read_array(img_path)
        mask = audit.read_array(mask_path)
        pal = audit.palette_report(mask)

        axes[row][0].imshow(image, cmap="gray" if image.ndim == 2 else None,
                            interpolation="nearest")
        axes[row][0].set_title(f"raw · {img_path.name}\n"
                               f"{image.shape[1]}x{image.shape[0]} · {image.dtype}",
                               fontsize=9)

        axes[row][1].imshow(mask, cmap="gray" if mask.ndim == 2 else None,
                            interpolation="nearest")
        axes[row][1].set_title(f"mask · {mask_path.name}\n"
                               f"{pal['n_unique_colours']} unique colours · "
                               f"quantized={pal['quantized']}", fontsize=9)

        top = pal["top_colours"]
        heights = [e["fraction"] for e in top]
        bar_colours = [
            tuple(np.clip(np.array(e["colour"] * 3 if len(e["colour"]) == 1
                                   else e["colour"], dtype=float) / 255.0, 0, 1))
            for e in top
        ]
        axes[row][2].bar(range(len(top)), heights, color=bar_colours,
                         edgecolor="0.4", linewidth=0.5)
        axes[row][2].set_yscale("log")
        axes[row][2].set_xticks(range(len(top)))
        axes[row][2].set_xticklabels([str(e["colour"]) for e in top],
                                     rotation=90, fontsize=6)
        axes[row][2].set_ylabel("pixel fraction (log)")
        axes[row][2].set_title("mask colour histogram, top "
                               f"{len(top)}", fontsize=9)

        for ax in axes[row][:2]:
            ax.set_xticks([])
            ax.set_yticks([])

    plt.tight_layout()
    plt.show()

## Checks

Nothing in this project runs locally, so this cell is where the step is
declared correct or not. It asserts that the audit actually audited: every
folder paired at least one image with a mask, no folder produced a mask with
zero unique colours (which would mean an unreadable or empty mask), and every
folder carries a mode verdict of exactly `A` or `B`. It also re-reads
`reports/audit.json` from disk, because the file — not the in-memory object —
is what later steps consume.

In [ ]:
import json

checks = []


def check(name, ok, detail=""):
    checks.append((name, bool(ok)))
    print(f"{'PASS' if ok else 'FAIL'}  {name}{'  -- ' + detail if detail else ''}")


check("audit.json written", json_path.is_file(), str(json_path))
check("audit.md written", md_path.is_file(), str(md_path))

on_disk = json.loads(json_path.read_text())
check("audit.json re-reads as JSON", isinstance(on_disk.get("datasets"), dict),
      f"{len(on_disk.get('datasets', {}))} folders")
check("no folder failed to audit", not on_disk.get("failures"),
      ", ".join(on_disk.get("failures", {})) or "none")
check("every expected dataset audited",
      set(PATHS["expected_datasets"]) <= set(on_disk["datasets"]),
      f"audited: {sorted(on_disk['datasets'])}")

for name, d in on_disk["datasets"].items():
    check(f"{name}: >0 matched pairs", d["pairing"]["n_pairs"] > 0,
          f"{d['pairing']['n_pairs']} pairs from {d['pairing']['n_images']} images "
          f"/ {d['pairing']['n_masks']} masks")
    n_col = d["palette"]["n_unique_colours"]["min"]
    check(f"{name}: >0 unique mask colours", n_col is not None and n_col > 0,
          f"min {n_col} colours per mask")
    check(f"{name}: mode verdict present", d["mode"]["inferred"] in ("A", "B"),
          f"MODE {d['mode']['inferred']} — {d['mode']['reason']}")

failed = [n for n, ok in checks if not ok]
print(f"\n{len(checks) - len(failed)}/{len(checks)} checks passed")
if failed:
    raise AssertionError("failed checks: " + ", ".join(failed))

## Push the audit back to the repo

`reports/audit.json` is the contract for every later step: mask mode, pairing
convention and boundary fractions are read from it and never re-derived or
assumed. It only becomes that contract once it is committed, so this cell
stages `reports/`, `configs/` and `notebooks/` — never data, checkpoints or
outputs — and pushes with the same `GH_TOKEN` secret the bootstrap used.

In [ ]:
from scripts.push_results import push_results

push_results("step 1: dataset audit", paths=PATHS)